# Financial Sentiment: Classical ML vs. Generative SLM

## Assignment Overview

You will compare two approaches to financial sentiment classification on the
Financial PhraseBank dataset:

1. **Logistic regression + TF-IDF** (a classical ML baseline)
2. **Microsoft Phi-3-mini** (a ~3.8B parameter generative small language model)
   used in zero-shot mode

Most of the code is provided. Your work is concentrated in **four critical-thinking
tasks** where the decisions you make are more important than the code you write.

### Your Tasks

| # | Task | Where | Estimated Time |
|---|---|---|---|
| 3A | Design a prompt for Phi-3-mini (iterate on a naive starting point) | Part 3 | 45–60 min |
| 3B | Write a parser for Phi-3-mini's free-text outputs | Part 3 | 30–45 min |
| 4.4 | Design and implement an **original** annotation dimension with a testable hypothesis | Part 4 | 60–90 min |
| 4.5 | Hand-annotate ~20 divergence examples with prose explanations | Part 4 | 90–120 min |
| 4.7 | Write a 1–2 sentence prose explanation for each row of Table 5 | Part 4 | 30–45 min |

### Deliverables

1. **This notebook**, completed end-to-end with all tasks filled in.
2. A **2–3 page Word document** report (template provided as
   `phi3_sentiment_report_template.docx`).

### How to Run

Open this notebook in Google Colab. Set **Runtime → Change runtime type → T4 GPU**.
Run cells top-to-bottom. Phi-3-mini is downloaded once (~7 GB) and cached;
full-test-set inference (Stage B in Part 3) takes about **12 minutes** on a T4.
Plan your iteration time accordingly — the dev loop (Stage A) runs in under a
minute so you can iterate your prompt and parser quickly there.

### Time Budget (5–7 hours across the week)

- Reading + setup: 30 min
- Task 3A + 3B (prompt + parser iteration): 1.5–2 hrs
- Task 4.4 (original dimension + hypothesis + code): 1–1.5 hrs
- Task 4.5 + 4.7 (hand annotation + Table 5 prose): 1.5–2 hrs
- Report writing: 1.5–2 hrs

### Grading

| Component | Weight |
|---|---|
| Prompt iteration quality and iteration log | 20% |
| Parser quality | 10% |
| Original dimension (design + hypothesis + operationalization) | 20% |
| Per-example annotation prose (hand annotation + Table 5) | 20% |
| Written report | 30% |

**The emphasis is critical thinking over programming.** A well-reasoned
sub-optimal prompt with a strong iteration log beats a lucky optimal prompt
with no explanation.


---
## Setup

Install dependencies, import everything, verify the GPU, and log in to
HuggingFace. Run every cell in this section once before proceeding.


### Install Required Libraries

Installs the libraries this notebook uses. You'll see a pip warning about
`fsspec` / `gcsfs` conflicts — that's expected and doesn't affect anything
here.


In [ ]:
!pip install transformers "datasets==2.19.0" accelerate scikit-learn --quiet --no-warn-conflicts


### Import Libraries

All imports live in this single cell. The constants at the bottom
(`LABEL_NAMES`, `UNPARSEABLE`, `RANDOM_SEED`) are used throughout the
notebook — don't change them.


In [ ]:
import os
import re
from collections import Counter
from typing import Callable, Iterable, Sequence

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from huggingface_hub import login
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer


# Canonical label mapping. Every other mapping is derived from this.
LABEL_NAMES: dict[int, str] = {0: "negative", 1: "neutral", 2: "positive"}
LABEL_IDS: dict[str, int] = {name: idx for idx, name in LABEL_NAMES.items()}
TARGET_NAMES: list[str] = [LABEL_NAMES[i] for i in range(len(LABEL_NAMES))]

# Sentinel for Phi-3 outputs that can't be parsed into a known label.
UNPARSEABLE: int = -1

# Global seed for reproducibility.
RANDOM_SEED: int = 42


### Verify GPU Access

Phi-3-mini needs a GPU to be usable. If the cell below prints `False`, go to
**Runtime → Change runtime type → T4 GPU** and restart.


In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"CUDA available: True | Device: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("WARNING: No GPU. Phi-3-mini will be unusable on CPU. Switch runtime to T4 GPU.")


### HuggingFace Login

Phi-3-mini is a public model and doesn't strictly require login, but
authenticating avoids anonymous rate limits. Paste a token from
https://huggingface.co/settings/tokens when prompted.


In [ ]:
login()  # Paste your access token when prompted.


---
## Part 1: Data Loading and Exploration

This section is fully provided. Read through the cells and look at the
outputs — in particular, note the class imbalance in Table 1, which
motivates the `class_weight='balanced'` choice in Part 2.

### 1.1 — Load the Dataset

We use the `sentences_75agree` subset of Financial PhraseBank. The `75agree`
subset includes sentences where at least 75% of human annotators agreed on
the label, which keeps genuinely ambiguous sentences in the data — useful
for the Part 4 error analysis.


In [ ]:
DATASET_NAME = "takala/financial_phrasebank"
DATASET_CONFIG = "sentences_75agree"

try:
    dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, trust_remote_code=True)
except Exception as err:
    raise RuntimeError(
        f"Failed to load {DATASET_NAME}/{DATASET_CONFIG}. "
        "Check your internet connection and that datasets==2.19.0 is installed."
    ) from err

train_split = dataset["train"]
print(f"Total examples: {len(train_split)}")
print(f"Fields:         {train_split.column_names}")
print(f"Label mapping:  {LABEL_NAMES}")
print(f"\nSample sentence: {train_split['sentence'][0]}")
print(f"Sample label:    {LABEL_NAMES[train_split['label'][0]]}")


### 1.2 — Train/Test Split

80/20 split with stratification. Stratification preserves the class balance
in both splits — without it, the 12%-negative class could drift between
train and test and distort the model comparison.


In [ ]:
sentences: list[str] = list(train_split["sentence"])
labels: list[int] = list(train_split["label"])

X_train, X_test, y_train, y_test = train_test_split(
    sentences,
    labels,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=labels,
)

print(f"Training set: {len(X_train)} examples")
print(f"Test set:     {len(X_test)} examples")


### 1.3 — Class Distribution (Table 1)

Table 1 reports class counts and percentages in each split. In
`sentences_75agree` the neutral class dominates (~59%) while negative is
smallest (~12%). The train and test percentages should be nearly identical
because of stratification.


In [ ]:
def class_distribution_table(
    y_train: Sequence[int],
    y_test: Sequence[int],
) -> pd.DataFrame:
    """Count+percentage breakdown of class labels across two splits."""
    train_counts = Counter(y_train)
    test_counts = Counter(y_test)
    n_train, n_test = len(y_train), len(y_test)

    rows = []
    for idx in sorted(LABEL_NAMES):
        rows.append({
            "Class":       LABEL_NAMES[idx],
            "Train Count": train_counts[idx],
            "Train %":     f"{train_counts[idx] / n_train * 100:.1f}%",
            "Test Count":  test_counts[idx],
            "Test %":      f"{test_counts[idx] / n_test * 100:.1f}%",
        })
    return pd.DataFrame(rows)


table1 = class_distribution_table(y_train, y_test)
print("=== Table 1: Class Distribution ===")
print(table1.to_string(index=False))
print(f"\nTotal train: {len(y_train)} | Total test: {len(y_test)}")


---
## Part 2: Logistic Regression Baseline

This section is fully provided. It trains the classical baseline that you
will compare against Phi-3-mini in Part 4.

### 2.1 — TF-IDF Features

Unigrams + bigrams, vocabulary capped at 10,000. The vectorizer is fit on
**training text only** — fitting on `X_train + X_test` would leak test-set
word-distribution information into the features.


In [ ]:
def build_tfidf_features(
    X_train_text: Sequence[str],
    X_test_text: Sequence[str],
    max_features: int = 10_000,
    ngram_range: tuple[int, int] = (1, 2),
) -> tuple[TfidfVectorizer, np.ndarray, np.ndarray]:
    """Fit TF-IDF on training text; transform both splits."""
    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)
    X_train_tfidf = vectorizer.fit_transform(X_train_text)
    X_test_tfidf = vectorizer.transform(X_test_text)
    return vectorizer, X_train_tfidf, X_test_tfidf


vectorizer, X_train_tfidf, X_test_tfidf = build_tfidf_features(X_train, X_test)
print(f"TF-IDF matrix shape (train): {X_train_tfidf.shape}")
print(f"TF-IDF matrix shape (test):  {X_test_tfidf.shape}")


### 2.2 — Train the Logistic Regression Model

`class_weight='balanced'` reweights the loss so each class contributes
equally regardless of frequency. Without it, a model that predicted
"neutral" for everything would score ~59% accuracy — high enough to hide
the fact that it learned nothing.


In [ ]:
def train_logistic_regression(
    X_train_features: np.ndarray,
    y_train_labels: Sequence[int],
    *,
    class_weight: str | dict | None = "balanced",
    max_iter: int = 1000,
    random_state: int = RANDOM_SEED,
) -> LogisticRegression:
    """Fit a logistic regression classifier with the given training features."""
    clf = LogisticRegression(
        class_weight=class_weight,
        max_iter=max_iter,
        random_state=random_state,
    )
    clf.fit(X_train_features, y_train_labels)
    return clf


lr_model = train_logistic_regression(X_train_tfidf, y_train)
print("Logistic regression model trained.")


### 2.3 — Evaluate the LR Model

The `evaluate_predictions` helper defined here will be reused for Phi-3-mini
in Part 3. Its `sentinel` parameter drops unparseable predictions from
**both** `y_pred` and `y_true`, which is how we guarantee both models are
scored on the same examples (even though Phi-3 might fail to produce a
parseable label on some inputs, while LR never does).

Look at the per-class precision, recall, and F1 in the output. These
numbers are your **LR Table 3** for the report.


In [ ]:
def evaluate_predictions(
    y_true: Sequence[int],
    y_pred: Sequence[int],
    model_name: str,
    *,
    sentinel: int | None = None,
) -> dict[str, float | int]:
    """Compute accuracy/macro-F1 and print a classification report.

    If ``sentinel`` is provided, predictions equal to that value are dropped
    (along with their corresponding true labels) before metrics are computed.
    """
    y_true_arr = np.asarray(y_true)
    y_pred_arr = np.asarray(y_pred)

    if sentinel is not None:
        valid_mask = y_pred_arr != sentinel
        n_filtered = int((~valid_mask).sum())
        y_true_arr = y_true_arr[valid_mask]
        y_pred_arr = y_pred_arr[valid_mask]
    else:
        n_filtered = 0

    accuracy = accuracy_score(y_true_arr, y_pred_arr)
    macro_f1 = f1_score(y_true_arr, y_pred_arr, average="macro")

    print(f"{model_name} — Accuracy: {accuracy:.4f}")
    print(f"{model_name} — Macro F1: {macro_f1:.4f}")
    if n_filtered:
        print(f"{model_name} — Filtered {n_filtered} unparseable prediction(s)")
    print()
    print(classification_report(
        y_true_arr, y_pred_arr,
        target_names=TARGET_NAMES,
        zero_division=0,
    ))

    return {
        "accuracy":    accuracy,
        "macro_f1":    macro_f1,
        "n_evaluated": int(len(y_true_arr)),
        "n_filtered":  n_filtered,
    }


lr_preds = lr_model.predict(X_test_tfidf)
lr_metrics = evaluate_predictions(y_test, lr_preds, "Logistic Regression")


### 2.4 — Save LR Predictions

Create a shared DataFrame holding test sentences, true labels, and LR
predictions. Phi-3 predictions will be added in Part 3.


In [ ]:
results_df = pd.DataFrame({
    "sentence":   X_test,
    "true_label": y_test,
    "lr_pred":    lr_preds,
})

print(f"Results DataFrame created: {len(results_df)} rows")
results_df.head()


---
## Part 3: Zero-Shot Inference with Phi-3-mini

In this section you'll use **Microsoft Phi-3-mini** (specifically
`microsoft/Phi-3-mini-4k-instruct`), a 3.8B-parameter generative language
model, to classify the same test sentences the LR model saw.

### What Phi-3-mini is (briefly)

- **Architecture:** decoder-only Transformer (Llama-style).
- **Parameters:** 3.8B — small by current LLM standards, but orders of
  magnitude larger than the LR model's vocabulary.
- **Training:** pretrained on ~3.3T tokens of filtered web and synthetic data;
  instruction-tuned to follow natural-language requests.
- **Zero-shot:** we do **not** train or fine-tune it on Financial PhraseBank.
  Any domain knowledge comes from its pretraining corpus.

### How generation is different from classification

Classifier-head models (BERT-style) map tokens → logits → `argmax` → label.
That's deterministic and the output is always one of three known values.
Phi-3-mini generates **free text** — it could say "positive", "The sentiment is
clearly negative", "Neither positive nor negative", or something else
entirely. You are responsible for:

1. **Prompting** it to produce parseable output.
2. **Parsing** whatever it does produce into our canonical label set.

Those are your Tasks 3A and 3B below.

### 3.1 — Load Phi-3-mini

Downloads the model (~7 GB the first time; cached afterwards) and loads it
onto the GPU in fp16. Takes 1–3 minutes. Review the output to confirm the
model is on `cuda`.


In [ ]:
PHI3_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"


def load_phi3(model_name: str = PHI3_MODEL_NAME):
    """Load Phi-3-mini for causal generation on the available device."""
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16,
        )
    except OSError as err:
        raise RuntimeError(
            f"Failed to load {model_name}. Verify network access and that "
            f"`huggingface_hub.login()` succeeded. Original error: {err}"
        ) from err

    # Decoder-only models need left-padding for batched generation.
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model.eval()
    return model, tokenizer


phi3_model, phi3_tokenizer = load_phi3()
print(f"Model loaded: {PHI3_MODEL_NAME}")
print(f"Model device: {phi3_model.device}")
print(f"Model dtype:  {phi3_model.dtype}")


---
## **STUDENT TASK 3A — Design Your Prompt**

Iterate on the prompt template below. The starting version (v0) is
deliberately mediocre — your job is to observe its failure modes on the dev
set (Stage A, a few cells down) and improve it.

### What to do

1. Run **everything below this task** once with the v0 prompt. Look at the
   raw model outputs printed in Stage A.
2. Identify at least **three specific failure modes** (e.g., "model adds
   'The sentiment is…' prefix", "model hedges with 'mostly positive'",
   "model misclassifies sentences with negation").
3. Edit `PROMPT_TEMPLATE` to address the failure modes. Re-run Stage A.
   Repeat until satisfied.
4. Keep **all iterations** in `PROMPT_ITERATION_LOG` (do not delete old
   versions). This log is graded.

### Tips (not a checklist — pick what's relevant to what you observe)

- Constrain the output format (`"Respond with exactly one word: positive,
  neutral, or negative."`)
- Add 1–3 few-shot examples with diverse labels
- Add a role or domain frame (`"You are a financial analyst..."`)
- Specify how to handle ambiguity explicitly
- Tell the model what NOT to do (e.g., don't explain, don't hedge)

### Prompt-fix vs. parser-fix

When you see a failure, you can fix it in **the prompt** (Task 3A) or in
**the parser** (Task 3B). They aren't mutually exclusive. A clean prompt
that constrains output usually beats a complex parser — but sometimes
parser-side fixes are easier. Note which one you chose for each failure
mode in your iteration log.

### Grading (20% of total)

- Final prompt achieves reasonable dev-set accuracy (target: >65%)
- Iteration log shows **≥3 distinct iterations** with clear rationale
- Each iteration explicitly connects a change to an observed failure mode


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# PROMPT TEMPLATE — edit this
# ──────────────────────────────────────────────────────────────────────
# Must contain exactly one "{sentence}" placeholder. The inference helper
# passes the raw sentence into this placeholder.

PROMPT_TEMPLATE = """Classify the sentiment of this financial sentence as positive, neutral, or negative.

Sentence: {sentence}

Sentiment:"""

# ──────────────────────────────────────────────────────────────────────
# PROMPT ITERATION LOG — update this as you iterate
# ──────────────────────────────────────────────────────────────────────
# Keep every iteration. Copy-paste the old prompt into the log before
# editing PROMPT_TEMPLATE above.

PROMPT_ITERATION_LOG = """
v0 — Starting point (provided):
    Prompt: Generic classification prompt, no output constraint.
    Dev accuracy: (run Stage A to measure)
    Observed failures: (fill in what you see)

v1 — (your first iteration):
    Prompt: (copy your new prompt here)
    Change: (what did you change vs. v0?)
    Rationale: (what failure mode does this address?)
    Dev accuracy: 
    Observed failures remaining:

v2 — (your second iteration):
    ...
"""


---
## **STUDENT TASK 3B — Write the Output Parser**

The parser below takes one raw model output string and returns one of
`"positive"`, `"neutral"`, `"negative"`, or `""` (empty means unparseable,
which becomes `UNPARSEABLE` downstream).

The v0 parser is a simple substring match. It has a predictable failure
mode you will almost certainly see: **if the model says "not positive" it
matches "positive"**. Fix it, or fix the prompt to prevent the model from
saying that in the first place.

### What to do

- Run Stage A with the v0 parser. Look at any outputs that get mis-parsed.
- Improve `parse_model_output` OR constrain the prompt so the parser is
  sufficient.
- Don't over-engineer. Twenty lines of nested regex is usually the wrong
  answer — it's a sign the prompt should be doing more of the work.

### Grading (10% of total)

- Parses correctly on your final dev-set outputs
- Robust to at least one failure mode you observed (not a total rewrite,
  but clear evidence you iterated)


In [ ]:
def parse_model_output(text: str) -> str:
    """Extract a sentiment label from a raw model output string.

    Returns one of 'positive', 'neutral', 'negative', or '' (empty string
    means the output couldn't be parsed into a known label).
    """
    text_lower = text.lower().strip()

    # Naive v0: check each label as a substring, first match wins.
    # This will incorrectly handle "not positive" → matches "positive".
    for label in ("positive", "neutral", "negative"):
        if label in text_lower:
            return label

    return ""  # unparseable


### 3.2 — Inference Helper (provided)

This function runs batched generation through Phi-3-mini using the model's
chat template (the format Phi-3 was fine-tuned on). It returns one raw
string per input sentence. You don't need to modify it — but reading it
will help you understand what Phi-3 actually sees at inference time.


In [ ]:
def run_phi3_inference(
    sentences: Sequence[str],
    model,
    tokenizer,
    prompt_template: str,
    *,
    batch_size: int = 8,
    max_new_tokens: int = 20,
    verbose: bool = True,
) -> list[str]:
    """Run batched generation and return raw (unparsed) output strings.

    Each sentence is wrapped in ``prompt_template`` (which must contain a
    single ``{sentence}`` placeholder) and formatted via the model's chat
    template before tokenization.
    """
    raw_outputs: list[str] = []
    n = len(sentences)

    for start in range(0, n, batch_size):
        batch = list(sentences[start:start + batch_size])

        # Wrap each sentence in the student's prompt template, then apply
        # Phi-3's chat template so tokens match what the model was tuned on.
        formatted = []
        for s in batch:
            messages = [{"role": "user", "content": prompt_template.format(sentence=s)}]
            formatted.append(tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            ))

        inputs = tokenizer(
            formatted,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,  # greedy — deterministic
                pad_token_id=tokenizer.pad_token_id,
            )

        # Slice off the input tokens and decode only the new ones.
        new_tokens = outputs[:, inputs.input_ids.shape[1]:]
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        raw_outputs.extend(t.strip() for t in decoded)

        if verbose and (start // batch_size) % 5 == 0:
            done = min(start + batch_size, n)
            print(f"  Processed {done}/{n}")

    return raw_outputs


def parse_predictions(
    raw_outputs: Sequence[str],
    parser: Callable[[str], str],
    label_ids: dict[str, int] = LABEL_IDS,
    sentinel: int = UNPARSEABLE,
) -> tuple[list[int], list[tuple[int, str]]]:
    """Apply the student's parser to raw outputs, then map labels to integer IDs.

    Returns (predictions, failures). Failures are (index, raw_string) pairs
    for outputs the parser couldn't classify.
    """
    parsed: list[int] = []
    failures: list[tuple[int, str]] = []
    for i, raw in enumerate(raw_outputs):
        label = parser(raw)
        if label in label_ids:
            parsed.append(label_ids[label])
        else:
            parsed.append(sentinel)
            failures.append((i, raw))
    return parsed, failures


### 3.3 — Stage A: Dev Loop (iterate here)

This cell runs your current prompt + parser on **20 sampled test sentences**
and prints every raw model output. Re-run this cell each time you change
`PROMPT_TEMPLATE` (Task 3A) or `parse_model_output` (Task 3B). It takes
~30 seconds.

**What to look for in the output:**
- Raw outputs that don't cleanly say a single label word
- Parsed predictions that disagree with the true label (is the model wrong,
  or is the parser wrong?)
- A summary accuracy at the bottom — track this as you iterate


In [ ]:
DEV_SAMPLE_SIZE = 20

# Pick a reproducible subset of X_test for the dev loop.
rng = np.random.RandomState(RANDOM_SEED)
dev_indices = rng.choice(len(X_test), size=DEV_SAMPLE_SIZE, replace=False)
X_dev = [X_test[i] for i in dev_indices]
y_dev = [y_test[i] for i in dev_indices]

print(f"Running Phi-3-mini on {DEV_SAMPLE_SIZE} dev examples...")
print(f"Current prompt template:\n{'-' * 60}\n{PROMPT_TEMPLATE}\n{'-' * 60}\n")

dev_raw = run_phi3_inference(X_dev, phi3_model, phi3_tokenizer, PROMPT_TEMPLATE, verbose=False)

# Print every dev-set output so you can see what the model is doing.
print("=== Raw model outputs ===\n")
dev_parsed, dev_failures = parse_predictions(dev_raw, parse_model_output)
for i, (sentence, true_label, raw, pred_int) in enumerate(zip(X_dev, y_dev, dev_raw, dev_parsed)):
    true_name = LABEL_NAMES[true_label]
    pred_name = LABEL_NAMES[pred_int] if pred_int != UNPARSEABLE else "UNPARSEABLE"
    mark = "✓" if pred_int == true_label else "✗"
    print(f"[{i+1:02d}] {mark} True={true_name:>8} Pred={pred_name:>11} Raw={raw!r}")
    print(f"      Sentence: {sentence[:90]}{'...' if len(sentence) > 90 else ''}")
    print()

# Summary.
print("=" * 60)
print(f"Unparseable outputs: {len(dev_failures)}/{DEV_SAMPLE_SIZE}")
dev_metrics = evaluate_predictions(y_dev, dev_parsed, "Phi-3 (dev)", sentinel=UNPARSEABLE)


### Iteration checkpoint

Before moving to Stage B, confirm:

- [ ] You have at least **3 entries** in `PROMPT_ITERATION_LOG` above
- [ ] Dev accuracy is reasonable (target: **>65%**)
- [ ] Unparseable outputs on dev set are **≤2 / 20**

If any of these fail, go back and iterate Task 3A / Task 3B.

When you're satisfied, proceed to Stage B. Stage B takes ~12 minutes so
only run it **once** after you've locked in your prompt and parser.


### 3.4 — Stage B: Full Test-Set Inference (~12 min)

Once your prompt and parser are locked in, run this cell **once** to
generate Phi-3 predictions on the full test set. This is the set of
predictions that gets evaluated against LR in Part 4.


In [ ]:
print(f"Running Phi-3-mini on {len(X_test)} test examples...")
print("This takes ~12 minutes on a T4 GPU.\n")

phi3_raw_preds = run_phi3_inference(
    X_test, phi3_model, phi3_tokenizer, PROMPT_TEMPLATE
)
print(f"\nInference complete. {len(phi3_raw_preds)} raw outputs generated.")


### 3.5 — Parse and Evaluate

Apply your parser to the full-test-set outputs and report the metrics.
Any unparseable outputs are listed at the bottom — if there are many,
consider going back to iterate on the prompt/parser before moving on.


In [ ]:
phi3_preds_int, phi3_failures = parse_predictions(phi3_raw_preds, parse_model_output)

n_total = len(phi3_preds_int)
n_failed = len(phi3_failures)
print(f"Successfully parsed: {n_total - n_failed}/{n_total}")
print(f"Unparseable:         {n_failed}\n")

if phi3_failures:
    print("=== Unparseable outputs (first 10) ===")
    for idx, raw in phi3_failures[:10]:
        print(f"  Index {idx:4d} | Raw: {raw!r}")
    print()

phi3_metrics = evaluate_predictions(
    y_test, phi3_preds_int, "Phi-3-mini (zero-shot)", sentinel=UNPARSEABLE
)


### 3.6 — Add Phi-3 Predictions to `results_df`

Extends the shared DataFrame so Part 4 can compare both models side by side.


In [ ]:
results_df["phi3_pred"] = phi3_preds_int
results_df["phi3_raw"] = phi3_raw_preds

print(f"Columns in results_df: {list(results_df.columns)}")
results_df.head(10)


---
## Part 4: Comparative Analysis and Error Analysis

This is the heart of the assignment. The goal is to understand **why** and
**when** LR and Phi-3 diverge — not just which is more accurate.

### 4.1 — Prediction Agreement Matrix (Table 4)

A 3×3 cross-tabulation of LR predictions vs. Phi-3 predictions. The diagonal
is agreement; off-diagonal is divergence. Note that **neither axis is ground
truth** here — this table shows how the two models compare to each other.


In [ ]:
# Filter to rows where Phi-3 produced a parseable prediction.
valid_df = results_df[results_df["phi3_pred"] != UNPARSEABLE].copy()
print(f"Valid examples for comparison: {len(valid_df)}/{len(results_df)}")

agreement_matrix = confusion_matrix(
    valid_df["lr_pred"], valid_df["phi3_pred"],
    labels=sorted(LABEL_NAMES),
)

agreement_table = pd.DataFrame(
    agreement_matrix,
    index=[f"LR: {LABEL_NAMES[i]}" for i in sorted(LABEL_NAMES)],
    columns=[f"Phi-3: {LABEL_NAMES[i]}" for i in sorted(LABEL_NAMES)],
)

print("\n=== Table 4: Prediction Agreement Matrix ===")
print(agreement_table)

n_total = agreement_matrix.sum()
n_agree = int(np.trace(agreement_matrix))
print(f"\nAgreement rate:  {n_agree}/{n_total} ({n_agree / n_total:.1%})")
print(f"Divergence rate: {n_total - n_agree}/{n_total} ({(n_total - n_agree) / n_total:.1%})")


### 4.2 — Partition into Agreement and Divergence Sets

Split the valid examples into cases where the models agree vs. disagree,
and for each disagreement track which (if either) was correct.


In [ ]:
valid_df["agree"]        = valid_df["lr_pred"] == valid_df["phi3_pred"]
valid_df["lr_correct"]   = valid_df["lr_pred"]   == valid_df["true_label"]
valid_df["phi3_correct"] = valid_df["phi3_pred"] == valid_df["true_label"]

agreement_df  = valid_df[valid_df["agree"]].copy()
divergence_df = valid_df[~valid_df["agree"]].copy()

print(f"Agreement cases:  {len(agreement_df)}")
print(f"Divergence cases: {len(divergence_df)}")

div_lr_wins    = divergence_df[divergence_df["lr_correct"]   & ~divergence_df["phi3_correct"]]
div_phi3_wins  = divergence_df[divergence_df["phi3_correct"] & ~divergence_df["lr_correct"]]
div_both_wrong = divergence_df[~divergence_df["lr_correct"]  & ~divergence_df["phi3_correct"]]

print("\nWhen models disagree:")
print(f"  LR correct, Phi-3 wrong: {len(div_lr_wins)}")
print(f"  Phi-3 correct, LR wrong: {len(div_phi3_wins)}")
print(f"  Both wrong:              {len(div_both_wrong)}")


### 4.3 — Preview Divergence Examples

Read through a handful of divergence cases **before** defining your
annotation dimensions in Task 4.4. The patterns you see here should inform
what original dimension you propose.


In [ ]:
def preview_divergence(df: pd.DataFrame, n: int = 10) -> None:
    """Pretty-print up to n divergence cases."""
    print("=== Sample Divergence Cases ===\n")
    for _, row in df.head(n).iterrows():
        winner = ("LR" if row["lr_correct"]
                  else "Phi-3" if row["phi3_correct"]
                  else "Neither")
        print(f'"{row["sentence"]}"')
        print(f"  True: {LABEL_NAMES[row['true_label']]:>8} | "
              f"LR: {LABEL_NAMES[row['lr_pred']]:>8} | "
              f"Phi-3: {LABEL_NAMES[row['phi3_pred']]:>8} | "
              f"Correct: {winner}")
        print()


preview_divergence(divergence_df, n=10)


### 4.4 — Annotation Dimensions

Each sentence in the annotation sample will be coded against four dimensions
that characterize *what kind* of language it contains. Three are provided
(negation, hedging, domain specificity). The fourth is **your original
dimension** — Task 4.4 below.

Each dimension is a word or phrase list. The `make_term_coder` factory
compiles the list into a regex that matches whole words or whole phrases
(so "vs" won't match inside "versus", and "fell short" matches as a
two-word phrase).

The three required term lists:


In [ ]:
NEGATION_TERMS: tuple[str, ...] = (
    "not", "no", "never", "none", "neither", "nor", "nobody", "nothing",
    "nowhere", "cannot", "can't", "won't", "isn't", "aren't", "wasn't",
    "weren't", "doesn't", "don't", "didn't", "hasn't", "haven't", "hadn't",
    "wouldn't", "couldn't", "shouldn't", "without", "hardly", "barely",
    "scarcely",
)

HEDGE_TERMS: tuple[str, ...] = (
    "may", "might", "could", "would", "should", "possibly", "perhaps",
    "probably", "likely", "unlikely", "seems", "appear", "appears",
    "appeared", "suggest", "suggests", "suggested", "indicate", "indicates",
    "indicated", "expect", "expects", "expected", "anticipate",
    "anticipates", "anticipated", "believe", "believes", "believed",
    "assume", "assumes", "assumed", "approximately", "roughly", "around",
    "about", "somewhat", "generally", "typically", "usually", "often",
    "sometimes", "uncertain", "unclear",
)

DOMAIN_TERMS: tuple[str, ...] = (
    "ebitda", "write-down", "writedown", "impairment", "covenant", "dilution",
    "dilutive", "restructuring", "headwind", "headwinds", "margin", "charge",
    "charges", "goodwill", "amortization", "depreciation", "leverage",
    "liquidity", "solvency",
)


---
## **STUDENT TASK 4.4 — Design Your Original Dimension**

This is the highest-weight critical-thinking task in the assignment. You
will design an **original annotation dimension** — one that captures a
linguistic or structural property of sentences that you believe drives
LR vs. Phi-3 divergence.

### What you do

1. Read through ~30 divergence examples (extend `preview_divergence` above
   if needed). Look for patterns.
2. Propose a dimension. Name it. Write a **testable hypothesis** — a claim
   about which model will be more affected and **why, in terms of model
   architecture or training**.
3. Operationalize it. Typically a term list; alternatively a regex,
   length-threshold, or numeric-content rule.
4. Register your dimension by filling in the four fields below.

### Requirements

- **Distinct from the three required dimensions.** Don't rename an existing
  one. "Double negation" is just negation; "long hedges" is just hedging.
- **Testable hypothesis.** You must state in advance which model you
  expect to be hurt and why. State it **before** you look at the results.
- **Mechanism-linked.** The "why" should refer to how TF-IDF or Phi-3
  actually works (e.g. "TF-IDF can't compose features across positions",
  "Phi-3 learned X from pretraining corpus Y"). "It's hard" or "it's
  ambiguous" are **not** mechanism-linked.
- **Operationalizable.** A grader must be able to compute your dimension
  flag from the sentence alone.

### Examples

**STRONG example:** "Forward-looking / guidance language (`will`,
`forecast`, `projected`, `guidance`, `outlook`). Hypothesis: Phi-3 will
handle these better because its pretraining corpus includes large volumes
of forward-looking financial text where sentiment is inferred from the
direction of the projection, while LR treats forward-looking words as
sparse, high-dimensional features with little signal."

→ Named, testable, mechanism-linked, operationalizable.

**WEAK example 1:** "Complex sentences. Hypothesis: complex sentences are
harder."

→ Not operationalizable, not mechanism-linked, tautological.

**WEAK example 2:** "Negation combined with hedging. Hypothesis: this
combination is especially hard."

→ Not distinct from the required dimensions.

**WEAK example 3:** "Use of numbers. Hypothesis: numbers confuse both
models."

→ Operationalizable but not mechanism-linked, and "confuse both models"
isn't a directional prediction.

### Grading (20% of total)

- Dimension is **distinct** from the three required ones
- Hypothesis states a **direction** (which model suffers) and a
  **mechanism** (why)
- Term list / rule is **operationalizable** and contains **≥8 terms**
- Hypothesis is written **before** results are examined (honor system; we
  can tell from the quality of reasoning)


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# STUDENT TASK 4.4: fill in all four fields below.
# ──────────────────────────────────────────────────────────────────────

DIMENSION_NAME: str = ""  # e.g. "Forward-Looking Language"

DIMENSION_HYPOTHESIS: str = """
YOUR HYPOTHESIS HERE.

Must state:
  - Which model you expect to be more affected (LR or Phi-3)
  - In which direction (helped, hurt)
  - WHY, in terms of how that model actually works

Minimum two sentences, maximum a short paragraph.
"""

# Your term list (used by the regex coder below). Word-boundary matching is
# handled automatically — you just provide the list. Multi-word phrases are
# allowed (e.g. "fell short").
YOUR_DIMENSION_TERMS: tuple[str, ...] = (
    # Add 8 or more terms here.
)

# Brief justification of the term list (why these and not others).
DIMENSION_JUSTIFICATION: str = """
Explain why these specific terms operationalize your dimension well.
What does the list include, and what did you deliberately leave out?
"""


### 4.4 (continued) — Compile all Four Coders

The `make_term_coder` factory compiles each term list into a regex that
matches whole words/phrases (case-insensitive). The coders are registered
in a single `DIMENSION_CODERS` dict. Your original dimension is added
automatically if you've completed Task 4.4; otherwise a warning prints
and the notebook falls back to the three required dimensions.


In [ ]:
def make_term_coder(terms: Iterable[str]) -> Callable[[str], int]:
    """Return a function that flags sentences containing any of ``terms``.

    Matching is case-insensitive and uses word boundaries. Multi-word
    phrases (e.g. "fell short") are matched as whole phrases.
    """
    if not terms:
        raise ValueError("make_term_coder requires at least one term.")
    pattern = re.compile(
        r"\b(?:" + "|".join(re.escape(t) for t in terms) + r")\b",
        flags=re.IGNORECASE,
    )

    def coder(sentence: str) -> int:
        return int(pattern.search(sentence) is not None)

    return coder


# Required dimensions.
DIMENSION_CODERS: dict[str, Callable[[str], int]] = {
    "dim_negation":           make_term_coder(NEGATION_TERMS),
    "dim_hedging":            make_term_coder(HEDGE_TERMS),
    "dim_domain_specificity": make_term_coder(DOMAIN_TERMS),
}

DIMENSION_LABELS: list[tuple[str, str]] = [
    ("Negation",           "dim_negation"),
    ("Hedging",            "dim_hedging"),
    ("Domain Specificity", "dim_domain_specificity"),
]

# Your original dimension — registered only if Task 4.4 is complete.
if DIMENSION_NAME and YOUR_DIMENSION_TERMS:
    DIMENSION_CODERS["dim_original"] = make_term_coder(YOUR_DIMENSION_TERMS)
    DIMENSION_LABELS.append((DIMENSION_NAME, "dim_original"))
    print(f"Registered original dimension: {DIMENSION_NAME!r} "
          f"with {len(YOUR_DIMENSION_TERMS)} terms")
else:
    print("WARNING: Original dimension not yet defined (Task 4.4).")
    print("The notebook will run with only the three required dimensions.")


def apply_dimension_coders(
    df: pd.DataFrame,
    text_col: str = "sentence",
) -> pd.DataFrame:
    """Add one 0/1 column per coder to ``df`` and return it."""
    for col_name, coder in DIMENSION_CODERS.items():
        df[col_name] = df[text_col].apply(coder)
    return df


apply_dimension_coders(divergence_df)
apply_dimension_coders(agreement_df)
print(f"\nDimension columns: {list(DIMENSION_CODERS)}")


### 4.5 — Build the Annotation Sample and Export to CSV

Samples 20 divergence examples (supplements with agreement cases if
divergence < 20), applies all four coders, and exports to
`annotation_table.csv`. You'll fill in a `notes` column by hand in the next
task.


In [ ]:
N_DIVERGENCE = 20
N_AGREEMENT_SUPPLEMENT = 7


def build_annotation_sample(
    divergence_df: pd.DataFrame,
    agreement_df: pd.DataFrame,
    n_divergence: int = N_DIVERGENCE,
    n_supplement: int = N_AGREEMENT_SUPPLEMENT,
    random_state: int = RANDOM_SEED,
) -> pd.DataFrame:
    """Sample examples for annotation. Supplement with agreement cases if needed."""
    if len(divergence_df) >= n_divergence:
        sample = divergence_df.sample(n=n_divergence, random_state=random_state).copy()
        sample["group"] = "divergence"
        print(f"Sampling {n_divergence} divergence cases.")
        return sample
    div_part = divergence_df.copy()
    div_part["group"] = "divergence"
    supplement_n = min(n_supplement, len(agreement_df))
    sup_part = agreement_df.sample(n=supplement_n, random_state=random_state).copy()
    sup_part["group"] = "agreement"
    print(
        f"Divergence set small ({len(divergence_df)}) — supplementing with "
        f"{supplement_n} agreement cases."
    )
    return pd.concat([div_part, sup_part], ignore_index=False)


annotation_sample = build_annotation_sample(divergence_df, agreement_df)
apply_dimension_coders(annotation_sample)

# Add readable label columns.
annotation_sample["true_label_name"] = annotation_sample["true_label"].map(LABEL_NAMES)
annotation_sample["lr_pred_name"]    = annotation_sample["lr_pred"].map(LABEL_NAMES)
annotation_sample["phi3_pred_name"]  = annotation_sample["phi3_pred"].map(LABEL_NAMES)
annotation_sample["winner"] = np.where(
    annotation_sample["lr_correct"], "LR",
    np.where(annotation_sample["phi3_correct"], "Phi-3", "Neither"),
)
annotation_sample["notes"] = ""  # you fill this in

SAVE_COLUMNS = [
    "group", "sentence", "true_label_name", "lr_pred_name", "phi3_pred_name",
    "winner", "lr_correct", "phi3_correct",
    *DIMENSION_CODERS,
    "notes",
]

ANNOTATION_CSV_PATH = "/content/annotation_table.csv"
try:
    annotation_sample[SAVE_COLUMNS].to_csv(ANNOTATION_CSV_PATH, index=False)
    print(f"\nAnnotation table saved to {ANNOTATION_CSV_PATH}")
except OSError:
    fallback = "annotation_table.csv"
    annotation_sample[SAVE_COLUMNS].to_csv(fallback, index=False)
    print(f"\nAnnotation table saved to ./{fallback}")

print(f"Total rows: {len(annotation_sample)}")


---
## **STUDENT TASK 4.5 — Hand Annotate the 20 Examples**

Open `annotation_table.csv` (from the file browser in Colab: click the
folder icon on the left, then `content` → `annotation_table.csv` →
right-click → Download). Edit it in a spreadsheet, then re-upload as
`annotation_table_completed.csv` and run the next cell to read it back.

### What to fill in (the `notes` column)

For each row, write **1–3 sentences** explaining **why** you think the
models diverged. The automated dimension flags tell you *what* kind of
language the sentence contains; your note should explain *how* that
linguistic feature caused the failure **mechanistically**.

### Examples

**Good note (mechanism-linked):**
> "The word 'not' flips the sentiment of 'profitable' but TF-IDF treats
> the two as independent features. LR would need to have learned the
> specific bigram 'not profitable' during training to get this; it
> apparently hasn't, so it defaults to positive cues elsewhere in the
> sentence. Phi-3 attends from 'not' to 'profitable' in the same layer
> and correctly produces 'negative'."

**Weak note (descriptive only):**
> "The sentence has negation. LR got it wrong."

### Grading (20% of total)

- ≥ 15 of 20 notes include a **mechanism-level explanation** (not just
  "LR got it wrong" or "this is ambiguous")
- Notes reference the **specific words/phrases** in the sentence
- Notes are consistent with the dimension flags (if you say "hedging
  matters here" but the hedging flag is 0, explain the discrepancy)


### 4.5 (continued) — Read Back the Completed Annotations

After you fill in the `notes` column and re-upload the CSV, run this cell.
It reads the completed annotations back into the DataFrame so downstream
cells can use them.

**If you haven't re-uploaded yet**, this cell will print a warning and
continue with the unannotated sample — you can always re-run it later.


In [ ]:
COMPLETED_CSV_PATH_CANDIDATES = [
    "/content/annotation_table_completed.csv",
    "./annotation_table_completed.csv",
]

completed_path = next(
    (p for p in COMPLETED_CSV_PATH_CANDIDATES if os.path.exists(p)),
    None,
)

if completed_path:
    print(f"Reading hand-annotated notes from {completed_path}")
    completed = pd.read_csv(completed_path)
    # Merge the notes column back in (align on sentence text).
    notes_map = dict(zip(completed["sentence"], completed["notes"].fillna("")))
    annotation_sample["notes"] = annotation_sample["sentence"].map(notes_map).fillna("")
    n_annotated = (annotation_sample["notes"].str.strip() != "").sum()
    print(f"  Loaded {n_annotated}/{len(annotation_sample)} non-empty notes.")
else:
    print("No annotation_table_completed.csv found yet.")
    print("Download annotation_table.csv, fill in the notes column, and")
    print("re-upload as annotation_table_completed.csv, then re-run this cell.")


### 4.6 — Dimensional Analysis

For each dimension, computes the rate at which each model is correct when
the dimension is present vs. absent in the sentence. The `dimension_summary`
DataFrame is a required deliverable — include it in your report.

With n=20 annotated examples, subgroup sizes are small (5–15 per cell).
Treat observed differences as **suggestive patterns**, not statistically
significant findings.


In [ ]:
def _subset_correctness(subset: pd.DataFrame) -> tuple[int, float, int, float]:
    """(lr_correct_count, lr_rate, phi3_correct_count, phi3_rate)."""
    n = len(subset)
    if n == 0:
        return 0, 0.0, 0, 0.0
    lr_n   = int(subset["lr_correct"].sum())
    phi3_n = int(subset["phi3_correct"].sum())
    return lr_n, lr_n / n, phi3_n, phi3_n / n


def print_dimension_breakdown(df: pd.DataFrame) -> None:
    """Present/absent breakdown of model correctness per dimension."""
    for dim_name, dim_col in DIMENSION_LABELS:
        present = df[df[dim_col] == 1]
        absent  = df[df[dim_col] == 0]
        print(f"\n{'=' * 50}\n  {dim_name}\n{'=' * 50}")
        for label, subset in (("Present", present), ("Absent ", absent)):
            n = len(subset)
            if n == 0:
                print(f"  {label} (n=0): No examples")
                continue
            lr_n, lr_rate, p3_n, p3_rate = _subset_correctness(subset)
            print(f"  {label} (n={n}): LR {lr_n}/{n} ({lr_rate:.0%}), "
                  f"Phi-3 {p3_n}/{n} ({p3_rate:.0%})")


def build_dimension_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Tidy summary table for the written report."""
    rows = []
    for dim_name, dim_col in DIMENSION_LABELS:
        for value, label in ((1, "Present"), (0, "Absent")):
            subset = df[df[dim_col] == value]
            if len(subset) == 0:
                continue
            lr_n, lr_rate, p3_n, p3_rate = _subset_correctness(subset)
            rows.append({
                "Dimension":    dim_name,
                "Value":        label,
                "n":            len(subset),
                "LR Correct":   lr_n,
                "LR Rate":      f"{lr_rate:.0%}",
                "Phi-3 Correct": p3_n,
                "Phi-3 Rate":   f"{p3_rate:.0%}",
            })
    return pd.DataFrame(rows)


print_dimension_breakdown(annotation_sample)
dimension_summary = build_dimension_summary(annotation_sample)
print("\n\n=== Dimensional Summary Table (include in report) ===")
print(dimension_summary.to_string(index=False))


### 4.7 — Curate Table 5

Selects a handful of illustrative divergence examples grouped by pattern.
The programmatic selection below is a starting point — see Task 4.7 below
for what you need to add.


In [ ]:
def curate_table5(
    df: pd.DataFrame,
    dimensions: Sequence[tuple[str, str, int]] | None = None,
    max_sentence_chars: int = 100,
) -> pd.DataFrame:
    """Build Table 5 by pulling up to n examples per dimension, deduplicated."""
    if dimensions is None:
        # Default: pull 2-3 examples per dimension in DIMENSION_LABELS order.
        n_per_dim = {"Negation": 3, "Hedging": 3, "Domain Specificity": 3}
        # Add the original dimension with n=2 if it's registered.
        dimensions = [
            (name, col, n_per_dim.get(name, 2))
            for (name, col) in DIMENSION_LABELS
        ]

    seen_indices: set = set()
    rows = []
    for pattern_name, dim_col, n in dimensions:
        candidates = df[(df[dim_col] == 1) & (~df.index.isin(seen_indices))]
        for idx, row in candidates.head(n).iterrows():
            seen_indices.add(idx)
            winner = ("LR" if row["lr_correct"]
                      else "Phi-3" if row["phi3_correct"]
                      else "Neither")
            sentence = row["sentence"]
            if len(sentence) > max_sentence_chars:
                sentence = sentence[:max_sentence_chars] + "..."
            rows.append({
                "Sentence":    sentence,
                "True":        LABEL_NAMES[row["true_label"]],
                "LR Pred":     LABEL_NAMES[row["lr_pred"]],
                "Phi-3 Pred":  LABEL_NAMES[row["phi3_pred"]],
                "Correct":     winner,
                "Pattern":     pattern_name,
                "Your Note":   row.get("notes", ""),
            })
    return pd.DataFrame(rows)


table5 = curate_table5(annotation_sample)
print("=== Table 5: Curated Divergence Examples ===\n")
with pd.option_context("display.max_colwidth", 80):
    print(table5.to_string(index=False))


---
## **STUDENT TASK 4.7 — Write Prose for Each Table 5 Row**

If you completed Task 4.5 (hand annotation), the `Your Note` column of
Table 5 is already populated with your notes. Review Table 5 above and
check:

1. **Every row has a note in `Your Note`.** If any row's note is empty or
   very short, go back to `annotation_table_completed.csv` and expand it —
   Table 5 is what the grader reads most carefully.
2. **Notes in Table 5 are 1–2 sentences each** and mechanism-linked
   (see the guidance in Task 4.5).
3. **Notes reference the specific pattern label.** If the row's `Pattern`
   is "Negation", the note should reference how negation drove the
   divergence, not just describe the sentence.

Copy Table 5 (with your notes) into the report — it is **Section 3 Table
1** of your write-up.

### Grading (part of the 20% Annotation Prose total with Task 4.5)

- Every Table 5 row has a non-trivial note
- Notes are consistent between the CSV and Table 5
- At least one note per dimension


---
### Overall Metrics Summary (Table 2)

Side-by-side accuracy and macro-F1 for both models. Include this in your
report as Table 2.


In [ ]:
table2 = pd.DataFrame([
    {"Model": "Logistic Regression",
     "Accuracy": f"{lr_metrics['accuracy']:.4f}",
     "Macro F1": f"{lr_metrics['macro_f1']:.4f}",
     "n":        lr_metrics["n_evaluated"]},
    {"Model": "Phi-3-mini (zero-shot)",
     "Accuracy": f"{phi3_metrics['accuracy']:.4f}",
     "Macro F1": f"{phi3_metrics['macro_f1']:.4f}",
     "n":        phi3_metrics["n_evaluated"]},
])

print("=== Table 2: Overall Metrics ===")
print(table2.to_string(index=False))
print(
    f"\nNote: Phi-3-mini was evaluated on {phi3_metrics['n_evaluated']} "
    f"examples (out of {len(y_test)}); {phi3_metrics['n_filtered']} outputs "
    "were unparseable. LR was evaluated on all examples. Both models' Table 2 "
    "metrics use the same filter for fair comparison in Table 4."
)


---
## Part 5: Written Report

Your written report is a separate **2–3 page Word document** submitted
alongside this notebook. A starter template is provided as
`phi3_sentiment_report_template.docx`.

### Report Focus 

The report is organized around **model characteristics, decision points,
and results analysis** — not an exhaustive methods description.

### Required Structure (matches the Word template)

#### Section 1 — Model Characteristics (½ page)

Brief descriptions of the two models being compared, written with enough
specificity that a reader could predict where each one would struggle:

- **Logistic regression with TF-IDF**: feature representation, vocabulary
  size, key hyperparameters, what the `class_weight='balanced'` choice
  does.
- **Phi-3-mini**: parameter count, architecture family, training data in
  broad strokes (no need to reproduce the tech report — one or two
  sentences on what gives it domain competence). Note that you used it
  zero-shot.

#### Section 2 — Decision Points (1 page)

This is the distinctive section of the report. Walk the reader
through the substantive choices you made:

1. **Prompt design.** Show your final `PROMPT_TEMPLATE`. Summarize the
   iteration log — what failure modes you saw with v0, what you changed
   in each iteration, and why. One paragraph is sufficient; the iteration
   log itself can be included as an appendix if it's long.

2. **Parser design.** Brief (1–2 sentences) — describe whether you pushed
   the fix upstream to the prompt or kept it in the parser, and why.

3. **Original annotation dimension.** State your dimension name, the
   hypothesis you wrote *before* seeing results, and a 1-sentence
   justification of the term list. This must match what's in Task 4.4 of
   the notebook.

#### Section 3 — Results Analysis (1–1½ pages)

- **Table 2** — overall metrics (from the notebook).
- **Table 3** — per-class precision/recall/F1 for both models (from the
  `classification_report` outputs in Sections 2.3 and 3.5).
- **Table 4** — prediction agreement matrix (from Section 4.1).
- **Table 5** — curated divergence examples with your prose notes (from
  Section 4.7).
- **Dimensional summary** (from Section 4.6).

Then, in **prose**:

- Which dimension was the strongest predictor of divergence?
- Did your original dimension's hypothesis hold? If not, what did the
  data show instead? (A rejected hypothesis with clear analysis is
  worth full credit.)
- Interpret the test metrics (overall, per-class). What insights can you extract from them? 
- Connect at least **one** observed pattern to a specific model mechanism
  (feature independence in TF-IDF, attention in Phi-3, limits of
  word-list dimensions, etc.).

### What NOT to include

- A reproduction of the dataset description
- A line-by-line methods walkthrough (that's what the notebook is for)
- Unprocessed raw output dumps

### Page limit

2–3 pages, tables inclusive. Single-spaced, 11-pt minimum. Tables and a
short appendix (e.g. the prompt iteration log) don't count toward the
limit.
